# Azure 聊天补全示例（预览）

> 注意：openai 库已有更新版本。请参阅 https://github.com/openai/openai-python/discussions/742

本示例将介绍如何使用 Azure OpenAI 服务进行聊天补全。

## 设置


首先，安装必要的依赖。

In [ ]:
! pip install "openai>=0.28.1,<1.0.0"

为了让后续部分正常运行，我们首先需要进行一些设置。先从 `api_base` 和 `api_version` 开始。要找到你的 `api_base`，请前往 https://portal.azure.com, 找到你的资源，然后在“Resource Management” -> “Keys and Endpoints”下查找“Endpoint”的值。

In [ ]:
import os
import openai

In [ ]:
openai.api_version = '2023-05-15'
openai.api_base = '' # Please add your endpoint here

接下来需要设置 `api_type` 和 `api_key`。我们可以从门户获取密钥，也可以通过 Microsoft Active Directory 身份验证获取。根据所用方式，`api_type` 为 `azure` 或 `azure_ad`。

### 设置：门户
我们先看看如何从门户获取密钥。前往 https://portal.azure.com, 找到你的资源，然后在“Resource Management” -> “Keys and Endpoints”下查找其中一个“Keys”值。

In [ ]:
openai.api_type = 'azure'
openai.api_key = os.environ["OPENAI_API_KEY"]


> 注意：在本示例中，我们通过在代码中设置变量来配置库使用 Azure API。在开发中，建议改为设置环境变量：

```
OPENAI_API_BASE
OPENAI_API_KEY
OPENAI_API_TYPE
OPENAI_API_VERSION
```

### （可选）设置：Microsoft Active Directory 身份验证
现在来看看如何通过 Microsoft Active Directory 身份验证获取密钥。如果你想使用 Active Directory 身份验证而不是门户中的密钥，请取消注释以下代码。

In [ ]:
# from azure.identity import DefaultAzureCredential

# default_credential = DefaultAzureCredential()
# token = default_credential.get_token("https://cognitiveservices.azure.com/.default")

# openai.api_type = 'azure_ad'
# openai.api_key = token.token

令牌在一段时间内有效，之后便会过期。为确保每个请求都携带有效令牌，你可以通过挂接 requests.auth 来刷新即将过期的令牌：

In [ ]:
import typing
import time
import requests
if typing.TYPE_CHECKING:
    from azure.core.credentials import TokenCredential

class TokenRefresh(requests.auth.AuthBase):

    def __init__(self, credential: "TokenCredential", scopes: typing.List[str]) -> None:
        self.credential = credential
        self.scopes = scopes
        self.cached_token: typing.Optional[str] = None

    def __call__(self, req):
        if not self.cached_token or self.cached_token.expires_on - time.time() < 300:
            self.cached_token = self.credential.get_token(*self.scopes)
        req.headers["Authorization"] = f"Bearer {self.cached_token.token}"
        return req

session = requests.Session()
session.auth = TokenRefresh(default_credential, ["https://cognitiveservices.azure.com/.default"])

openai.requestssession = session

## 部署
在本节中，我们将使用 `gpt-35-turbo` 模型创建一个部署，之后可用它来创建聊天补全。

### 部署：手动创建
让我们使用 `gpt-35-turbo` 模型创建一个部署。前往 https://portal.azure.com, 找到你的资源，然后在“Resource Management” -> “Model deployments”下创建一个新的 `gpt-35-turbo` 部署。 

In [ ]:
deployment_id = '' # Fill in the deployment id from the portal here

### 创建聊天补全
现在向该部署发送一个示例聊天补全请求。

In [ ]:
# For all possible arguments see https://platform.openai.com/docs/api-reference/chat-completions/create
response = openai.ChatCompletion.create(
    deployment_id=deployment_id,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Knock knock."},
        {"role": "assistant", "content": "Who's there?"},
        {"role": "user", "content": "Orange."},
    ],
    temperature=0,
)

print(f"{response.choices[0].message.role}: {response.choices[0].message.content}")

我们也可以流式获取响应。


In [ ]:
response = openai.ChatCompletion.create(
    deployment_id=deployment_id,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Knock knock."},
        {"role": "assistant", "content": "Who's there?"},
        {"role": "user", "content": "Orange."},
    ],
    temperature=0,
    stream=True
)

for chunk in response:
    if len(chunk.choices) > 0:
        delta = chunk.choices[0].delta

        if "role" in delta.keys():
            print(delta.role + ": ", end="", flush=True)
        if "content" in delta.keys():
            print(delta.content, end="", flush=True)